<a href="https://colab.research.google.com/github/sr00t3d/colab/blob/main/wp_audit_xmlrpc_rest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile wp_audit.sh
#!/usr/bin/env bash
set -euo pipefail

TARGET="${1:-https://wordpress.org}"

fetch_nvd_cves() {
  local wp_ver="$1"
  local encoded="wordpress%20$wp_ver"

  curl -s --connect-timeout 8 --max-time 25 --retry 2 --retry-delay 1 \
    "https://services.nvd.nist.gov/rest/json/cves/2.0?keywordSearch=$encoded&keywordExactMatch" |
    jq -r '.vulnerabilities[]? |
      "- " + .cve.id + ": " +
      (.cve.descriptions[] | select(.lang=="en").value) +
      " (https://osv.dev/vulnerability/" + .cve.id + ")"'
}

echo "[+] Alvo: $TARGET"

SITE_VERSION=$(curl -skL "$TARGET" | grep -oP 'WordPress\s+\K[0-9]+\.[0-9]+(\.[0-9]+)?' | head -n1 || true)
echo "[+] Versão detectada: ${SITE_VERSION:-nao detectada}"

if [[ -n "${SITE_VERSION:-}" ]]; then
  echo "[+] Consultando CVEs no NVD..."
  nvd_cves=$(fetch_nvd_cves "$SITE_VERSION")
  if [[ -n "${nvd_cves//[[:space:]]/}" ]]; then
    echo "$nvd_cves"
  else
    echo "Nenhuma CVE encontrada para a versão $SITE_VERSION"
  fi
fi

echo "[+] Testando XML-RPC methods..."
XMLRPC_GET_METHODS=$(curl -sLk -X POST "$TARGET/xmlrpc.php" \
  -H "Content-Type: text/xml" \
  --data '<?xml version="1.0"?>
  <methodCall><methodName>system.listMethods</methodName></methodCall>' || true)

declare -a XMLRPC_METHODS_LIST
while IFS= read -r METHOD; do
  XMLRPC_METHODS_LIST+=("$METHOD")
done < <(echo "$XMLRPC_GET_METHODS" | grep -oP '(?<=<string>).*?(?=</string>)' || true)

if printf '%s\n' "${XMLRPC_METHODS_LIST[@]:-}" | grep -q "^system.multicall$"; then
  echo "[!] system.multicall ATIVO - risco de brute force / amplificacao"
else
  echo "[-] system.multicall nao identificado"
fi

echo "[+] Testando enumeracao REST users..."
REST_USERS=$(curl -sk "$TARGET/wp-json/wp/v2/users" | jq -r '.[]? | "\(.id): \(.name) (\(.slug))"' 2>/dev/null || true)

if [[ -n "${REST_USERS//[[:space:]]/}" ]]; then
  echo "$REST_USERS"
  echo "$REST_USERS" | grep -q '^1:' && echo "[!] ID 1 exposto"
  echo "$REST_USERS" | grep -Eiq 'admin|administrator|root' && echo "[!] Slug sensivel detectado"
else
  echo "[-] Sem exposicao direta de usuarios ou endpoint restrito"
fi

Writing wp_audit.sh


In [ ]:
# @title Configuração do Alvo
target_url = "https://flowgames.gg" # @param {type:"string"}
!bash wp_audit.sh {target_url}

[+] Alvo: https://flowgames.gg
[+] Versão detectada: 6.9.7
[+] Consultando CVEs no NVD...
Nenhuma CVE encontrada para a versão 6.9.7
[+] Testando XML-RPC methods...
[-] system.multicall nao identificado
[+] Testando enumeracao REST users...
42: Alvaro Neto (alvaro-neto)
22: Bruno Micali (mica)
33: César Martins (cesarmartins)
35: David Tso (davidtso)
55: Equipe Flow Games (joaostocco)
29: Erik Brasil (erik)
51: Felipe Cardoso (cardoso)
39: Flow Games (flow-games)
34: Hugo Carvalho (hugocarvalho)
45: Jeff Kayo (jeff-kayo)
